# 実務への載せ方 — Shadow比較で新旧を並べる

出典：第07章 03節／パス：`07_production/01_shadow.ipynb`／パッケージ：numpy・pandas・scikit-learn・joblib／実行時間の目安：約1分

このノート単体で実行できる。`input/shadow_base.csv`（現行）と `input/shadow_new.csv`（新着）を読み込む。欠品時は内蔵生成へ切り替わる。
過去データのholdoutでPoCの比較をし、合格した候補を本番切替の前に新旧並行記録（Shadow）して不一致点を洗い出す手順である。

In [1]:
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RS = 0


def make_shadow_data(n: int, income_shift: float = 0.0, seed: int = 70, cut: float = 300.0) -> pd.DataFrame:
    r = np.random.default_rng(seed)
    income = np.clip(r.normal(450.0 + income_shift, 150.0, size=n), 150.0, 1500.0)
    past_due = np.clip(r.poisson(0.4, size=n), 0, 5)
    use_rate = r.uniform(0.0, 1.0, size=n)
    housing = r.choice(["rent", "own", "company"], size=n, p=[0.55, 0.35, 0.10])
    # 低年収×高利用率の層だけ危険度が跳ね上がる（線形では表しにくい区切り）。区切りの年収は景気で動く
    risky = (income < cut) & (use_rate > 0.6)
    logit = -3.0 + 0.9 * past_due - 0.002 * (income - 450.0) + 0.2 * (housing == "rent") + 2.5 * risky
    y = (r.uniform(size=n) < 1.0 / (1.0 + np.exp(-logit))).astype(int)
    return pd.DataFrame({"income": income, "past_due": past_due,
                         "use_rate": use_rate, "housing": housing, "default": y})


try:
    base = pd.read_csv("input/shadow_base.csv")
    new = pd.read_csv("input/shadow_new.csv")
    print("Inputを使用: input/shadow_base.csv, input/shadow_new.csv")
except FileNotFoundError:
    base = make_shadow_data(6000, income_shift=0.0)
    # 景気悪化で年収全体が下がり、危険な層の区切りも下へ動いた想定
    new = make_shadow_data(2000, income_shift=-120.0, seed=71, cut=180.0)
    print("Inputが無いため合成データを生成")
print(f"現行学習: {len(base)}件（貸倒率{base['default'].mean():.3f}） / 新着: {len(new)}件（貸倒率{new['default'].mean():.3f}）")

Inputを使用: input/shadow_base.csv, input/shadow_new.csv
現行学習: 6000件（貸倒率0.112） / 新着: 2000件（貸倒率0.141）


In [2]:
# 現行（線形）と新候補（勾配ブースティング）を過去データの75%で育て、残り25%（holdout）でPoCの比較をする
Xb, yb = base.drop(columns=["default"]), base["default"]
X_tr, X_ho, y_tr, y_ho = train_test_split(Xb, yb, test_size=0.25, stratify=yb, random_state=RS)
pre = ColumnTransformer([
    ("num", StandardScaler(), ["income", "past_due", "use_rate"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["housing"]),
])
# 前処理は clone で別々に持たせる（同じオブジェクトを共有すると、後の fit で前処理が上書きされる）
old = make_pipeline(clone(pre), LogisticRegression(max_iter=5000)).fit(X_tr, y_tr)
new_model = make_pipeline(
    clone(pre), HistGradientBoostingClassifier(max_iter=100, random_state=RS)).fit(X_tr, y_tr)
ap_old_ho = average_precision_score(y_ho, old.predict_proba(X_ho)[:, 1])
ap_new_ho = average_precision_score(y_ho, new_model.predict_proba(X_ho)[:, 1])
print(f"[PoC・holdout] 現行PR-AUC={ap_old_ho:.4f} / 候補PR-AUC={ap_new_ho:.4f}（差 {ap_new_ho - ap_old_ho:+.4f}）")
print("PoCの判定（事前基準：+0.03以上）: " + ("合格→Shadowへ進む" if ap_new_ho - ap_old_ho >= 0.03 else "不合格→見送り"))
joblib.dump(new_model, "candidate_pipe.joblib")
print("候補モデルを保存: candidate_pipe.joblib")

[PoC・holdout] 現行PR-AUC=0.3312 / 候補PR-AUC=0.3920（差 +0.0608）
PoCの判定（事前基準：+0.03以上）: 合格→Shadowへ進む
候補モデルを保存: candidate_pipe.joblib


In [3]:
# Shadow記録：新着データで両者を動かし、判定だけ残す（業務への影響なし）
Xn, yn = new.drop(columns=["default"]), new["default"].to_numpy()
p_old = old.predict_proba(Xn)[:, 1]
p_new = new_model.predict_proba(Xn)[:, 1]
print(f"[新着] 現行PR-AUC={average_precision_score(yn, p_old):.4f} / 候補PR-AUC={average_precision_score(yn, p_new):.4f}")
# 不一致の洗い出し：否決が覆る件（顧客影響が直接出る）
deny_old = p_old >= 0.30
deny_new = p_new >= 0.30
flip = int(((deny_old != deny_new)).sum())
print(f"否決の不一致: {flip}件（{flip / len(yn):.1%}）")
print(f"旧のみ否決: {int((deny_old & ~deny_new).sum())}件 / 新のみ否決: {int((~deny_old & deny_new).sum())}件")
assert np.isfinite(p_new).all()

[新着] 現行PR-AUC=0.3322 / 候補PR-AUC=0.3165
否決の不一致: 346件（17.3%）
旧のみ否決: 102件 / 新のみ否決: 244件


In [4]:
# 不一致の内訳：どちらが正しかったか（答え合わせは新着の実績で）
both_deny = deny_old & deny_new
print(f"両方否決の貸倒率: {yn[both_deny].mean():.3f}")
only_new = ~deny_old & deny_new
print(f"新のみ否決の貸倒率: {yn[only_new].mean():.3f}（高いほど切替の根拠になる）")
only_old = deny_old & ~deny_new
print(f"旧のみ否決の貸倒率: {yn[only_old].mean():.3f}（高いほど候補が危険な申込を通す証拠になる）")
# 判定材料のまとめ：PoC（過去データ）の差が新着で再現したか
ap_old_new = average_precision_score(yn, p_old)
ap_new_new = average_precision_score(yn, p_new)
print(f"[改善結果] PR-AUCの差（候補−現行）: PoC・holdout {ap_new_ho - ap_old_ho:+.4f} → 新着 {ap_new_new - ap_old_new:+.4f}")
print("切替の判断材料：精度差＋不一致の内訳＋人手審査の余力を合わせて合議する")

両方否決の貸倒率: 0.380
新のみ否決の貸倒率: 0.201（高いほど切替の根拠になる）
旧のみ否決の貸倒率: 0.324（高いほど候補が危険な申込を通す証拠になる）
[改善結果] PR-AUCの差（候補−現行）: PoC・holdout +0.0608 → 新着 -0.0157
切替の判断材料：精度差＋不一致の内訳＋人手審査の余力を合わせて合議する


In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
_book_fig = Path("../../../books/python-machine-jitumu/assets/images")
FIG = str(_book_fig if _book_fig.is_dir() else Path("outputs"))
Path(FIG).mkdir(parents=True, exist_ok=True)
x = np.arange(2); w = 0.35
plt.figure()
plt.bar(x - w / 2, [ap_old_ho, ap_old_new], w, label="current (linear)")
plt.bar(x + w / 2, [ap_new_ho, ap_new_new], w, label="candidate (HGB)")
plt.xticks(x, ["PoC holdout (past)", "shadow (new)"]); plt.ylabel("PR-AUC"); plt.legend()
plt.title("current vs candidate: past holdout and new data")
plt.savefig(f"{FIG}/code-result-j07-01.png", dpi=150, bbox_inches="tight"); plt.close()
print("saved code-result-j07-01.png")

saved code-result-j07-01.png


## 読み取りどころ

- PoC（過去データのholdout）で勝った候補が、新着でも勝つとは限らないこと。Shadowはそれを業務に影響させずに確かめる段階であること。
- 新着データでの精度差だけでなく、不一致の件数と内訳が切替判断の材料になること。
- 新のみ否決の貸倒率が高ければ切替の根拠、旧のみ否決の貸倒率が高ければ候補が危険な申込を通す証拠になること。
- 前処理は clone で別々に持たせること。Shadow記録は業務に影響しないこと（判定だけ残し、実行は旧のまま）。